# CatBoost Curriculum (Negative -> Positive)

- Undersampling 없이 전체 데이터로 2단계 학습
- Stage 1: label=0 중심 (label=1 가중치 낮춤)
- Stage 2: label=1 보강 (label=1 가중치 높여 이어서 학습)

In [2]:
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold


@dataclass
class CurriculumResult:
    model: str
    oof_ap: float
    cv_mean: float
    cv_std: float


def _default_catboost_params(seed: int) -> Dict:
    return {
        'learning_rate': 0.05,
        'depth': 8,
        'l2_leaf_reg': 10,
        'loss_function': 'Logloss',
        'eval_metric': 'PRAUC',
        'random_seed': seed,
        'early_stopping_rounds': 50,
        'auto_class_weights': 'Balanced',
        'verbose': False,
    }


def _linear_weight_schedule(start: float, end: float, stages: int) -> np.ndarray:
    stages = max(1, int(stages))
    if stages == 1:
        return np.array([float(end)], dtype=np.float32)
    return np.linspace(float(start), float(end), num=stages, dtype=np.float32)


def fit_fold_negative_then_positive(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    *,
    seed: int,
    base_params: Optional[Dict] = None,
    stage1_iterations: int = 350,
    stage2_iterations: int = 850,
    stage1_positive_weight_start: float = 0.02,
    stage1_positive_weight_end: float = 0.2,
    stage1_warmup_stages: int = 4,
    stage2_positive_weight: float = 3.0,
    nl_confidence_floor: float = 0.15,
    nl_noise_weight: float = 0.25,
    use_hard_filter: bool = False,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    y_tr = np.asarray(y_tr)
    if set(np.unique(y_tr)) != {0, 1}:
        raise ValueError('y_tr must contain both labels 0 and 1 for binary classification.')

    neg_idx = np.where(y_tr == 0)[0]
    pos_idx = np.where(y_tr == 1)[0]

    if len(neg_idx) == 0 or len(pos_idx) == 0:
        raise ValueError('Both classes must be present in each fold train split.')

    params = _default_catboost_params(seed)
    if base_params:
        params.update(base_params)

    params_stage2 = dict(params)
    params_stage2['iterations'] = int(stage2_iterations)

    # Stage 1 warmup: 0 클래스는 전부 유지, 1 클래스 가중치를 점진적으로 증가
    stage1_total_iters = max(1, int(stage1_iterations))
    warmup_stages = max(1, int(stage1_warmup_stages))
    warmup_weights = _linear_weight_schedule(
        stage1_positive_weight_start,
        stage1_positive_weight_end,
        warmup_stages,
    )

    base_iters = stage1_total_iters // warmup_stages
    rem_iters = stage1_total_iters % warmup_stages

    m_prev = None
    for stage_idx, pos_w in enumerate(warmup_weights):
        iter_this_stage = base_iters + (1 if stage_idx < rem_iters else 0)
        iter_this_stage = max(1, int(iter_this_stage))

        w_stage1 = np.ones(len(y_tr), dtype=np.float32)
        w_stage1[pos_idx] = float(pos_w)

        params_stage1_step = dict(params)
        params_stage1_step['iterations'] = iter_this_stage

        m_step = CatBoostClassifier(**params_stage1_step)
        fit_kwargs = dict(
            X=X_tr,
            y=y_tr,
            sample_weight=w_stage1,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        if m_prev is not None:
            fit_kwargs['init_model'] = m_prev

        m_step.fit(**fit_kwargs)
        m_prev = m_step

    m1 = m_prev

    # Filtering: NL 결과 기반으로 각 샘플의 true-label 확신도 계산
    tr_prob = m1.predict_proba(X_tr)[:, 1]
    true_conf = np.where(y_tr == 1, tr_prob, 1.0 - tr_prob).astype(np.float32)
    noise_mask = true_conf < float(nl_confidence_floor)

    info = {
        'noise_ratio': float(np.mean(noise_mask)),
        'kept_ratio': float(np.mean(~noise_mask)),
        'stage1_pos_weight_start': float(stage1_positive_weight_start),
        'stage1_pos_weight_end': float(stage1_positive_weight_end),
        'stage1_warmup_stages': float(stage1_warmup_stages),
    }

    # PL phase 기본 클래스 가중치
    w_stage2 = np.ones(len(y_tr), dtype=np.float32)
    w_stage2[pos_idx] = float(stage2_positive_weight)

    if use_hard_filter:
        keep_mask = ~noise_mask
        # hard filter 시 한 클래스가 사라지면 전체 사용으로 fallback
        if np.unique(y_tr[keep_mask]).shape[0] == 2 and int(np.sum(keep_mask)) >= 64:
            X_pl = X_tr[keep_mask]
            y_pl = y_tr[keep_mask]
            w_pl = w_stage2[keep_mask]
        else:
            X_pl = X_tr
            y_pl = y_tr
            w_pl = w_stage2
            info['kept_ratio'] = 1.0
            info['noise_ratio'] = 0.0
    else:
        # down-weight: 노이즈 의심 샘플의 영향만 줄이고 전체 데이터는 유지
        conf_scaled = np.clip(
            (true_conf - float(nl_confidence_floor)) / max(1e-6, 1.0 - float(nl_confidence_floor)),
            0.0,
            1.0,
        )
        conf_weight = 0.2 + 0.8 * conf_scaled
        noise_weight = np.where(noise_mask, float(nl_noise_weight), 1.0).astype(np.float32)
        w_pl = w_stage2 * conf_weight.astype(np.float32) * noise_weight
        w_pl = np.clip(w_pl, 1e-3, None)
        X_pl = X_tr
        y_pl = y_tr

    m2 = CatBoostClassifier(**params_stage2)
    m2.fit(
        X_pl,
        y_pl,
        sample_weight=w_pl,
        eval_set=[(X_val, y_val)],
        init_model=m1,
        verbose=False,
    )

    val_pred = m2.predict_proba(X_val)[:, 1]
    test_pred = m2.predict_proba(X_test)[:, 1]
    return val_pred, test_pred, info


def run_cv_negative_then_positive(
    X: np.ndarray,
    y: np.ndarray,
    X_test: np.ndarray,
    *,
    seed: int = 42,
    n_splits: int = 5,
    base_params: Optional[Dict] = None,
    stage1_iterations: int = 350,
    stage2_iterations: int = 850,
    stage1_positive_weight_start: float = 0.02,
    stage1_positive_weight_end: float = 0.2,
    stage1_warmup_stages: int = 4,
    stage2_positive_weight: float = 3.0,
    nl_confidence_floor: float = 0.15,
    nl_noise_weight: float = 0.25,
    use_hard_filter: bool = False,
    model_name: str = 'CatBoost_NL_Filtering_PL',
) -> Tuple[CurriculumResult, np.ndarray, np.ndarray]:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.zeros(len(y), dtype=np.float32)
    test_p = np.zeros(len(X_test), dtype=np.float32)
    fold_aps = []

    print(f'=== {model_name} ===')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        val_pred, te_pred, fit_info = fit_fold_negative_then_positive(
            X_tr,
            y_tr,
            X_val,
            y_val,
            X_test,
            seed=seed + fold,
            base_params=base_params,
            stage1_iterations=stage1_iterations,
            stage2_iterations=stage2_iterations,
            stage1_positive_weight_start=stage1_positive_weight_start,
            stage1_positive_weight_end=stage1_positive_weight_end,
            stage1_warmup_stages=stage1_warmup_stages,
            stage2_positive_weight=stage2_positive_weight,
            nl_confidence_floor=nl_confidence_floor,
            nl_noise_weight=nl_noise_weight,
            use_hard_filter=use_hard_filter,
        )

        oof[val_idx] = val_pred.astype(np.float32)
        test_p += te_pred.astype(np.float32) / n_splits

        ap = float(average_precision_score(y_val, val_pred))
        fold_aps.append(ap)
        print(
            f'  Fold {fold} AP={ap:.5f} '
            + f'| train_pos_rate={np.mean(y_tr):.4%} val_pos_rate={np.mean(y_val):.4%} '
            + f'| warmup={fit_info["stage1_pos_weight_start"]:.3f}->{fit_info["stage1_pos_weight_end"]:.3f}'
            + f'/{int(fit_info["stage1_warmup_stages"])} '
            + f'| kept={fit_info["kept_ratio"]:.2%} noise={fit_info["noise_ratio"]:.2%}'
        )

    oof_ap = float(average_precision_score(y, oof))
    cv_mean = float(np.mean(fold_aps))
    cv_std = float(np.std(fold_aps))
    print(f'  OOF AP={oof_ap:.5f} | CV mean+-std={cv_mean:.5f} +- {cv_std:.5f}')

    result = CurriculumResult(
        model=model_name,
        oof_ap=oof_ap,
        cv_mean=cv_mean,
        cv_std=cv_std,
    )
    return result, oof, test_p

In [3]:
import os
import random

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ.setdefault('PYTHONHASHSEED', str(SEED))

print('pandas:', pd.__version__)
print('numpy :', np.__version__)

# 원본 데이터 그대로 로드 (값 변환/클리핑/파생변수 생성 없음)
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

TARGET = 'fraud_bool'
DROP_COLS = ['id', TARGET]

# One-Hot Encoding만 유지: object/category 컬럼만 인코딩
cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()
print('One-hot 대상 컬럼:', cat_cols)

if len(cat_cols) > 0:
    _ohe_tr = pd.get_dummies(train[cat_cols], dtype=np.int8)
    _ohe_te = pd.get_dummies(test[cat_cols], dtype=np.int8)
    _ohe_te = _ohe_te.reindex(columns=_ohe_tr.columns, fill_value=0)

    train = train.drop(columns=cat_cols)
    test = test.drop(columns=cat_cols)
    train = pd.concat([train, _ohe_tr], axis=1)
    test = pd.concat([test, _ohe_te], axis=1)

feature_cols = [c for c in train.columns if c not in DROP_COLS]
X = train[feature_cols].values
y = train[TARGET].values
X_test = test[feature_cols].values

print(f'Prepared: X={X.shape}, y={y.shape}, X_test={X_test.shape}')

pandas: 3.0.1
numpy : 2.4.3
One-hot 대상 컬럼: ['payment_type', 'employment_status', 'housing_status', 'application_source', 'device_os']
Prepared: X=(700000, 52), y=(700000,), X_test=(300000, 52)


## 사용 순서

아래 순서대로 실행하세요.

1. 위 데이터 로드 + One-Hot 셀 실행 (원본값 유지)
2. 그다음 아래 학습/제출 셀 실행

셀 실행 후 `X, y, X_test, SEED, test`가 준비됩니다.

In [5]:
# 현재 설정은 수동 sample_weight를 강하게 쓰므로 auto_class_weights는 끄는 편이 안정적
cb_params = dict(globals().get('CATBOOST_BEST_PARAMS', {}) or {})
cb_params['auto_class_weights'] = None

r, oof_pred, test_preds = run_cv_negative_then_positive(
    X, y, X_test,
    seed=SEED,
    n_splits=5,
    base_params=cb_params,
    stage1_iterations=320,
    stage2_iterations=900,
    stage1_positive_weight_start=0.02,  # 1 클래스 영향 아주 약하게 시작
    stage1_positive_weight_end=0.25,    # 1.0은 과도할 수 있어 완만하게 제한
    stage1_warmup_stages=4,             # (0.02 -> 0.10 -> 0.17 -> 0.25)
    stage2_positive_weight=2.2,         # FP 과다 시 3.0보다 완화
    nl_confidence_floor=0.18,
    nl_noise_weight=0.20,
    use_hard_filter=False,
    model_name='CatBoost_NegAll_PosGradual_PL_fix_v1',
)

print(r)
submission_out = pd.DataFrame({'id': test['id'], 'fraud': test_preds})
submission_out.to_csv('submission_netgative_curriculum.csv', index=False)
print('저장 완료: submission_netgative_curriculum.csv')
submission_out.head()

=== CatBoost_NegAll_PosGradual_PL_fix_v1 ===


KeyboardInterrupt: 

In [6]:
# Transformer 기반 탭уляр 분류기 (PyTorch)
# 실행 전 필요 시: pip install torch

import copy

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as e:
    raise ImportError("PyTorch가 필요합니다. 먼저 `pip install torch`를 실행해 주세요.") from e

from sklearn.preprocessing import StandardScaler


class ScalarFeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.feature_emb = nn.Parameter(torch.randn(n_features, d_token) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n_features] -> tokens: [batch, n_features, d_token]
        return x.unsqueeze(-1) * self.weight + self.bias + self.feature_emb


class TabularTransformer(nn.Module):
    def __init__(
        self,
        n_features: int,
        d_token: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        ff_mult: int = 4,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.tokenizer = ScalarFeatureTokenizer(n_features=n_features, d_token=d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_tok = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        z = torch.cat([cls, x_tok], dim=1)
        z = self.encoder(z)
        logits = self.head(z[:, 0, :]).squeeze(1)
        return logits


def _set_torch_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _predict_proba(model: nn.Module, X_np: np.ndarray, device: torch.device, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    probs = []
    ds = TensorDataset(torch.from_numpy(X_np.astype(np.float32)))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(device)
            p = torch.sigmoid(model(xb)).detach().cpu().numpy()
            probs.append(p)
    return np.concatenate(probs).astype(np.float32)


def fit_fold_transformer(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    *,
    seed: int,
    epochs: int = 16,
    batch_size: int = 1024,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    d_token: int = 64,
    n_heads: int = 4,
    n_layers: int = 2,
    dropout: float = 0.15,
    patience: int = 4,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    _set_torch_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    tr_ds = TensorDataset(
        torch.from_numpy(X_tr_s),
        torch.from_numpy(y_tr.astype(np.float32)),
    )
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, drop_last=False)

    model = TabularTransformer(
        n_features=X_tr_s.shape[1],
        d_token=d_token,
        n_heads=n_heads,
        n_layers=n_layers,
        dropout=dropout,
    ).to(device)

    pos = float(np.sum(y_tr == 1))
    neg = float(np.sum(y_tr == 0))
    pos_weight = torch.tensor([max(1.0, neg / max(1.0, pos))], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    best_state = None
    best_ap = -1.0
    best_epoch = 0
    wait = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for xb, yb in tr_dl:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += float(loss.item()) * xb.size(0)

        scheduler.step()

        val_pred = _predict_proba(model, X_val_s, device=device)
        val_ap = float(average_precision_score(y_val, val_pred))
        tr_loss = total_loss / max(1, len(X_tr_s))

        print(
            f"    [Transformer] epoch {ep:02d}/{epochs} "
            f"loss={tr_loss:.5f} val_AP={val_ap:.5f}"
        )

        if val_ap > best_ap:
            best_ap = val_ap
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    [Transformer] early stop at epoch {ep} (best={best_epoch})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_pred = _predict_proba(model, X_val_s, device=device)
    test_pred = _predict_proba(model, X_test_s, device=device)

    info = {
        'best_ap': float(best_ap),
        'best_epoch': float(best_epoch),
        'pos_weight': float(pos_weight.detach().cpu().numpy()[0]),
    }
    return val_pred, test_pred, info


def run_cv_tabular_transformer(
    X: np.ndarray,
    y: np.ndarray,
    X_test: np.ndarray,
    *,
    seed: int = 42,
    n_splits: int = 5,
    epochs: int = 16,
    batch_size: int = 1024,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    d_token: int = 64,
    n_heads: int = 4,
    n_layers: int = 2,
    dropout: float = 0.15,
    patience: int = 4,
    model_name: str = 'TabularTransformer_v1',
) -> Tuple[CurriculumResult, np.ndarray, np.ndarray]:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.zeros(len(y), dtype=np.float32)
    test_p = np.zeros(len(X_test), dtype=np.float32)
    fold_aps = []

    print(f'=== {model_name} ===')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        val_pred, te_pred, fit_info = fit_fold_transformer(
            X_tr,
            y_tr,
            X_val,
            y_val,
            X_test,
            seed=seed + fold,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            d_token=d_token,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
            patience=patience,
        )

        oof[val_idx] = val_pred
        test_p += te_pred / n_splits

        ap = float(average_precision_score(y_val, val_pred))
        fold_aps.append(ap)
        print(
            f"  Fold {fold} AP={ap:.5f} | best_epoch={int(fit_info['best_epoch'])} "
            + f"| pos_weight={fit_info['pos_weight']:.3f}"
        )

    oof_ap = float(average_precision_score(y, oof))
    cv_mean = float(np.mean(fold_aps))
    cv_std = float(np.std(fold_aps))
    print(f'  OOF AP={oof_ap:.5f} | CV mean+-std={cv_mean:.5f} +- {cv_std:.5f}')

    result = CurriculumResult(
        model=model_name,
        oof_ap=oof_ap,
        cv_mean=cv_mean,
        cv_std=cv_std,
    )
    return result, oof, test_p


# 실행
tr_result, tr_oof_pred, tr_test_preds = run_cv_tabular_transformer(
    X,
    y,
    X_test,
    seed=SEED,
    n_splits=5,
    epochs=16,
    batch_size=1024,
    lr=1e-3,
    weight_decay=1e-4,
    d_token=64,
    n_heads=4,
    n_layers=2,
    dropout=0.15,
    patience=4,
    model_name='TabularTransformer_v1',
)

print(tr_result)
submission_transformer = pd.DataFrame({'id': test['id'], 'fraud': tr_test_preds})
submission_transformer.to_csv('submission_transformer.csv', index=False)
print('저장 완료: submission_transformer.csv')
submission_transformer.head()

=== TabularTransformer_v1 ===


KeyboardInterrupt: 

In [ ]:
# 오답 유도 피처 분석: OOF 예측으로 FP/FN 분해
from sklearn.metrics import precision_recall_curve


def _best_f1_threshold(y_true: np.ndarray, y_score: np.ndarray) -> float:
    p, r, t = precision_recall_curve(y_true, y_score)
    if len(t) == 0:
        return 0.5
    f1 = 2.0 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    return float(t[int(np.argmax(f1))])


def _topk_threshold(y_true: np.ndarray, y_score: np.ndarray) -> float:
    k = int(np.sum(y_true == 1))
    if k <= 0:
        return 0.5
    k = min(k, len(y_score))
    # 양성 개수와 동일한 개수를 양성으로 예측하도록 임계값 설정
    return float(np.partition(y_score, -k)[-k])


def _feature_shift_table(df: pd.DataFrame, a_mask: np.ndarray, b_mask: np.ndarray, top_n: int = 20) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        a = df.loc[a_mask, col].astype(np.float32)
        b = df.loc[b_mask, col].astype(np.float32)

        if len(a) == 0 or len(b) == 0:
            continue

        mean_a = float(a.mean())
        mean_b = float(b.mean())
        gap = mean_a - mean_b

        pooled = pd.concat([a, b], axis=0)
        std = float(pooled.std(ddof=0))
        std_gap = gap / (std + 1e-12)

        rows.append(
            {
                'feature': col,
                'mean_A': mean_a,
                'mean_B': mean_b,
                'gap_A_minus_B': gap,
                'std_gap': std_gap,
                'abs_std_gap': abs(std_gap),
            }
        )

    out = pd.DataFrame(rows).sort_values('abs_std_gap', ascending=False)
    return out.head(top_n).reset_index(drop=True)


# 입력 준비
X_df = train[feature_cols].copy()
y_true = y.astype(int)
y_score = oof_pred.astype(np.float32)

thr_f1 = _best_f1_threshold(y_true, y_score)
thr_topk = _topk_threshold(y_true, y_score)

# class-imbalance 환경에서는 top-k 임계값도 같이 확인하는 것이 안전함
threshold = thr_topk
pred = (y_score >= threshold).astype(int)

fp_mask = (pred == 1) & (y_true == 0)
tn_mask = (pred == 0) & (y_true == 0)
fn_mask = (pred == 0) & (y_true == 1)
tp_mask = (pred == 1) & (y_true == 1)

print(f'threshold(f1-best)={thr_f1:.6f}')
print(f'threshold(top-k)  ={thr_topk:.6f}  <-- used')
print(f'FP={int(fp_mask.sum())}, FN={int(fn_mask.sum())}, TP={int(tp_mask.sum())}, TN={int(tn_mask.sum())}')

fp_confusion_features = _feature_shift_table(X_df, fp_mask, tn_mask, top_n=25)
fn_confusion_features = _feature_shift_table(X_df, fn_mask, tp_mask, top_n=25)

print('\n[FP vs TN] 오탐(정상인데 사기로 예측)을 유도하는 상위 피처')
display(fp_confusion_features)

print('\n[FN vs TP] 미탐(사기인데 정상으로 예측)을 유도하는 상위 피처')
display(fn_confusion_features)

# 후속 셀에서 재사용할 수 있도록 저장
error_analysis = {
    'threshold_f1': thr_f1,
    'threshold_topk': thr_topk,
    'used_threshold': threshold,
    'fp_idx': np.where(fp_mask)[0],
    'fn_idx': np.where(fn_mask)[0],
    'tp_idx': np.where(tp_mask)[0],
    'tn_idx': np.where(tn_mask)[0],
}

print('\nerror_analysis, fp_confusion_features, fn_confusion_features 생성 완료')

threshold(f1-best)=0.913786
threshold(top-k)  =0.921544  <-- used
FP=5829, FN=5829, TP=1813, TN=686529

[FP vs TN] 오탐(정상인데 사기로 예측)을 유도하는 상위 피처


,feature,mean_A,mean_B,gap_A_minus_B,std_gap,abs_std_gap
0,housing_status_BA,0.896380,0.158995,0.737385,1.985611,1.985611
1,device_os_windows,0.819180,0.254380,0.564800,1.289028,1.289028
2,req_credit_limit,1089.852417,507.064575,582.787842,1.203770,1.203770
3,credit_risk_score,212.597870,129.700562,82.897308,1.195052,1.195052
4,age_bucket,45.043747,33.506260,11.537487,0.962290,0.962290
5,is_session_persistent,0.208612,0.582471,-0.373859,-0.757308,0.757308
6,yearly_income,0.766255,0.559599,0.206656,0.712392,0.712392
7,dob_email_count_4w,6.214102,9.552744,-3.338642,-0.663545,0.663545
8,housing_status_BC,0.056099,0.376412,-0.320314,-0.662093,0.662093
9,device_os_linux,0.054898,0.337429,-0.282531,-0.598573,0.598573



[FN vs TP] 미탐(사기인데 정상으로 예측)을 유도하는 상위 피처


,feature,mean_A,mean_B,gap_A_minus_B,std_gap,abs_std_gap
0,housing_status_BA,0.475553,0.891892,-0.416339,-0.842033,0.842033
1,req_credit_limit,719.339539,1198.604492,-479.264954,-0.745096,0.745096
2,credit_risk_score,162.959167,223.343628,-60.384460,-0.740641,0.740641
3,device_os_windows,0.519643,0.814672,-0.295029,-0.599774,0.599774
4,age_bucket,39.210842,45.879757,-6.668915,-0.509165,0.509165
5,housing_status_BC,0.258020,0.056260,0.201760,0.495216,0.495216
6,is_session_persistent,0.395780,0.165472,0.230308,0.485787,0.485787
7,yearly_income,0.654263,0.779702,-0.125439,-0.467647,0.467647
8,dob_email_count_4w,7.975982,5.845560,2.130422,0.435829,0.435829
9,device_os_other,0.217705,0.063982,0.153722,0.399058,0.399058



error_analysis, fp_confusion_features, fn_confusion_features 생성 완료


In [ ]:
# 피처 그룹 단위 혼동 요약 + 오답 샘플 확인

def _feature_group_name(feature: str, cat_columns) -> str:
    for c in cat_columns:
        prefix = f'{c}_'
        if feature.startswith(prefix):
            return c
    return feature


def _group_confusion_table(conf_df: pd.DataFrame, cat_columns, top_n: int = 15) -> pd.DataFrame:
    t = conf_df.copy()
    t['group'] = t['feature'].map(lambda x: _feature_group_name(x, cat_columns))
    g = (
        t.groupby('group', as_index=False)['abs_std_gap']
        .sum()
        .sort_values('abs_std_gap', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return g


fp_group_summary = _group_confusion_table(fp_confusion_features, cat_cols, top_n=15)
fn_group_summary = _group_confusion_table(fn_confusion_features, cat_cols, top_n=15)

print('[FP 유도 그룹 Top]')
display(fp_group_summary)
print('[FN 유도 그룹 Top]')
display(fn_group_summary)

# 오답 샘플 원본 확인용 테이블 (필요 시 상위 N개만 확인)
err_idx = np.concatenate([error_analysis['fp_idx'][:20], error_analysis['fn_idx'][:20]])
err_view_cols = ['id', TARGET] + cat_cols
err_view_cols = [c for c in err_view_cols if c in train.columns]

print(f'오답 샘플 미리보기 행 수: {len(err_idx)}')
display(train.iloc[err_idx][err_view_cols].head(40))

[FP 유도 그룹 Top]


,group,abs_std_gap
0,housing_status,3.610712
1,device_os,2.466998
2,req_credit_limit,1.203770
3,credit_risk_score,1.195052
4,payment_type,1.091145
5,age_bucket,0.962290
6,is_session_persistent,0.757308
7,yearly_income,0.712392
8,dob_email_count_4w,0.663545
9,name_email_sim,0.586953


[FN 유도 그룹 Top]


,group,abs_std_gap
0,housing_status,2.010760
1,device_os,1.349644
2,req_credit_limit,0.745096
3,credit_risk_score,0.740641
4,payment_type,0.724327
5,age_bucket,0.509165
6,is_session_persistent,0.485787
7,yearly_income,0.467647
8,dob_email_count_4w,0.435829
9,is_home_phone_valid,0.350976


오답 샘플 미리보기 행 수: 40


,id,fraud_bool
15,15,0
101,101,0
383,383,0
536,536,0
619,619,0
663,663,0
680,680,0
834,834,0
876,876,0
1058,1058,0


In [ ]:
# 개선 실험: FP/FN 균형을 위한 파라미터 비교

cb_base = dict(globals().get('CATBOOST_BEST_PARAMS', {}) or {})
cb_base['auto_class_weights'] = None

trial_configs = [
    {
        'name': 'fix_conservative',
        'stage1_positive_weight_start': 0.02,
        'stage1_positive_weight_end': 0.20,
        'stage1_warmup_stages': 4,
        'stage2_positive_weight': 2.0,
        'nl_confidence_floor': 0.20,
        'nl_noise_weight': 0.18,
    },
    {
        'name': 'fix_balanced',
        'stage1_positive_weight_start': 0.02,
        'stage1_positive_weight_end': 0.25,
        'stage1_warmup_stages': 4,
        'stage2_positive_weight': 2.2,
        'nl_confidence_floor': 0.18,
        'nl_noise_weight': 0.20,
    },
    {
        'name': 'fix_aggressive',
        'stage1_positive_weight_start': 0.02,
        'stage1_positive_weight_end': 0.35,
        'stage1_warmup_stages': 5,
        'stage2_positive_weight': 2.8,
        'nl_confidence_floor': 0.15,
        'nl_noise_weight': 0.25,
    },
]

trial_rows = []
for cfg in trial_configs:
    print(f"\n=== trial: {cfg['name']} ===")
    rr, oof_tmp, _ = run_cv_negative_then_positive(
        X, y, X_test,
        seed=SEED,
        n_splits=5,
        base_params=cb_base,
        stage1_iterations=320,
        stage2_iterations=900,
        stage1_positive_weight_start=cfg['stage1_positive_weight_start'],
        stage1_positive_weight_end=cfg['stage1_positive_weight_end'],
        stage1_warmup_stages=cfg['stage1_warmup_stages'],
        stage2_positive_weight=cfg['stage2_positive_weight'],
        nl_confidence_floor=cfg['nl_confidence_floor'],
        nl_noise_weight=cfg['nl_noise_weight'],
        use_hard_filter=False,
        model_name=f"trial_{cfg['name']}",
    )

    thr = _topk_threshold(y.astype(int), oof_tmp.astype(np.float32))
    pred_tmp = (oof_tmp >= thr).astype(int)

    fp = int(((pred_tmp == 1) & (y == 0)).sum())
    fn = int(((pred_tmp == 0) & (y == 1)).sum())

    trial_rows.append(
        {
            'name': cfg['name'],
            'oof_ap': rr.oof_ap,
            'cv_mean': rr.cv_mean,
            'cv_std': rr.cv_std,
            'fp_at_topk': fp,
            'fn_at_topk': fn,
            'threshold_topk': float(thr),
            **cfg,
        }
    )

trial_df = pd.DataFrame(trial_rows).sort_values(['oof_ap', 'fn_at_topk'], ascending=[False, True]).reset_index(drop=True)
print('\n[Trial Summary]')
display(trial_df)

print('\n권장: oof_ap가 높고 fp/fn 균형이 맞는 설정 1개를 골라 5번 셀 파라미터에 반영')